# Verifikasi model acuan P1 dan P2

Notebook ini menghitung ulang biaya minimum dan audit WCC untuk tiga level jaringan. P1 memakai permintaan pasti. P2 memakai tiga skenario dengan satu keputusan buka gudang yang berlaku di ketiganya.

Pilih **Runtime → Run all** untuk menjalankan semua bagian. Notebook membaca data dan audit P1 dari versi acuan publik di GitHub. Pada salinan repo lokal, file yang cocok dengan versi acuan dipakai langsung. Bagian terakhir mengekspor audit P2 lengkap dalam CSV.

## 1. Muat data acuan

Data berisi rute, kapasitas, tarif, permintaan P1, dan parameter tiga skenario P2. Audit P1 berisi semua pasangan dari R1 sampai R39.

In [1]:
from csv import DictReader, DictWriter
from hashlib import sha256
from io import StringIO
from itertools import combinations, product
import json
from pathlib import Path
from urllib.request import urlopen

import numpy as np
from scipy.optimize import Bounds, LinearConstraint, linprog, milp

SNAPSHOT_ROOT = ("https://raw.githubusercontent.com/kiittkiidd/"
                 "Lampiran-Model-Matematika-Tesis/dc893d2e0aff4d1e9c9d2efe0b13d1af4488b3bb/")
DATA_PATH = "case/Data_Bersama.json"
AUDIT_PATH = "math_model_acuan/P1/Audit_WCC_Pasangan.csv"
SNAPSHOT_SHA256 = {
    DATA_PATH: "56f9774f5472da9a6b517349f716b46f3ec9de553044384fe2a3f77cf9e0b903",
    AUDIT_PATH: "4c2b3ecd3d99128e59ff21dbcc413243f8d877f7c2391cec2864ccbd0b6290a3",
}


def read_reference(path):
    for directory in (Path.cwd(), *Path.cwd().parents):
        candidate = directory / path
        if candidate.is_file():
            payload = candidate.read_bytes()
            if sha256(payload).hexdigest() == SNAPSHOT_SHA256[path]:
                return payload.decode("utf-8")
    with urlopen(SNAPSHOT_ROOT + path, timeout=30) as response:
        payload = response.read()
    if sha256(payload).hexdigest() != SNAPSHOT_SHA256[path]:
        raise ValueError(f"Isi data acuan berbeda: {path}")
    return payload.decode("utf-8")


def read_p1_pairs():
    return list(DictReader(StringIO(read_reference(AUDIT_PATH))))


DATA = json.loads(read_reference(DATA_PATH))
PRODUCTS = DATA["produk"]
WAREHOUSES = range(1, 5)
CUSTOMERS = range(1, 7)
HIGH_IN = DATA["level"]["H"]["jalur_masuk"]
BALANCES = sorted({(g, p) for _, g, p in HIGH_IN})
DEMAND_KEYS = list(product(CUSTOMERS, PRODUCTS))
SCENARIOS = DATA["P2"]["skenario"]
print("Tingkat jaringan:", ", ".join(DATA["level"]))
print("Skenario P2:", ", ".join(s["nama"] for s in SCENARIOS))

Tingkat jaringan: L, M, H
Skenario P2: Sepi, Sedang, Ramai


## 2. Bentuk matriks model

R1–R6 adalah kapasitas pabrik, R7–R17 neraca gudang, R18–R35 permintaan, dan R36–R39 kapasitas gudang. Tiga blok P2 memakai kolom keputusan buka gudang yang sama. Setiap blok punya kolom arus sendiri.

In [2]:
def route_matrices(inbound, outbound):
    """Bentuk 39 posisi baris pada urutan R1–R39 yang dipakai P1."""
    flows = [("x", *a) for a in inbound] + [("z", *a) for a in outbound]
    a = np.zeros((39, len(flows)))
    warehouse_coefficient = np.zeros((39, 4))
    for col, (kind, i, j, p) in enumerate(flows):
        if kind == "x":
            a[i - 1, col] = 1
            a[6 + BALANCES.index((j, p)), col] = 1
        else:
            a[6 + BALANCES.index((i, p)), col] = -1
            a[17 + DEMAND_KEYS.index((j, p)), col] = 1
            a[34 + i, col] = 1
    warehouse_coefficient[35:, :] = -np.diag(DATA["kapasitas_gudang"])
    return a, warehouse_coefficient


def rhs(demand):
    values = [demand.get(f"{j},{p}", 0.0) for j, p in DEMAND_KEYS]
    return np.array(DATA["kapasitas_pabrik"] + [0] * 11 + values + [0] * 4, dtype=float)


def operating_costs(inbound, outbound):
    return np.array(
        [DATA["tarif_masuk_per_palet"][i - 1][g - 1] for i, g, _ in inbound]
        + [
            DATA["tarif_keluar_per_palet"][g - 1][j - 1]
            + DATA["tarif_penanganan_per_palet"][g - 1][PRODUCTS.index(p)]
            for g, j, p in outbound
        ]
    )


def expanded_rows(a, gy, demands):
    """Susun baris skenario dengan satu set y yang dipakai bersama."""
    n = a.shape[1]
    matrix = np.zeros((39 * len(demands), 4 + n * len(demands)))
    row_upper = np.concatenate([rhs(demand) for demand in demands])
    row_lower = row_upper.copy()
    for scenario_index, _ in enumerate(demands):
        start = 39 * scenario_index
        block = slice(start, start + 39)
        flow_cols = slice(4 + scenario_index*n, 4 + (scenario_index+1)*n)
        matrix[block, :4] = gy
        matrix[block, flow_cols] = a
        row_lower[start:start+6] = -np.inf
        row_lower[start+35:start+39] = -np.inf
    return matrix, row_lower, row_upper

## 3. Hitung optimum dengan dua metode

Metode pertama mencoba 16 pola buka gudang dan menyelesaikan program linear arus untuk setiap skenario. Metode kedua menyelesaikan MILP gabungan. Biaya buka masuk sekali dan biaya operasi setiap skenario dikalikan peluangnya.

In [3]:
def enumerate_warehouses(a, gy, costs, demands, probabilities):
    """Enumerasi 16 pola y; selesaikan satu LP arus untuk setiap skenario."""
    choices = {}
    upper = np.r_[0:6, 35:39]
    for opened in product((0, 1), repeat=4):
        scenario_costs = []
        for demand in demands:
            residual = rhs(demand) - gy @ opened
            result = linprog(
                costs, A_ub=a[upper], b_ub=residual[upper],
                A_eq=a[6:35], b_eq=residual[6:35], bounds=(0, None),
                method="highs",
            )
            if result.status == 2:
                break
            if not result.success:
                raise RuntimeError(result.message)
            scenario_costs.append(result.fun)
        if len(scenario_costs) == len(demands):
            opening = float(np.dot(DATA["biaya_buka_gudang"], opened))
            choices[opened] = (
                opening + float(np.dot(probabilities, scenario_costs)),
                opening,
                scenario_costs,
            )
    return choices


def extensive_form(a, gy, costs, demands, probabilities):
    """Selesaikan MILP deterministik ekuivalen."""
    n = len(costs)
    matrix, row_lower, row_upper = expanded_rows(a, gy, demands)
    objective = np.r_[DATA["biaya_buka_gudang"], np.concatenate(
        [probability * costs for probability in probabilities]
    )]
    result = milp(
        objective, integrality=np.r_[np.ones(4), np.zeros(n*len(demands))],
        bounds=Bounds(0, np.r_[np.ones(4), np.full(n*len(demands), np.inf)]),
        constraints=LinearConstraint(matrix, row_lower, row_upper),
        options={"mip_rel_gap": 0},
    )
    if not result.success:
        raise RuntimeError(result.message)
    return result.fun, matrix


def serialise_best(choices):
    value = min(record[0] for record in choices.values())
    tied = [y for y, record in choices.items() if np.isclose(record[0], value)]
    chosen = tied[0]
    return chosen, {
        "biaya_objektif": value,
        "pilihan_optimal": [[f"G{g}" for g, take in zip(WAREHOUSES, y) if take] for y in tied],
        "biaya_buka": choices[chosen][1],
        "biaya_operasi": choices[chosen][2],
        "pola_gudang_layak": len(choices),
    }

## 4. Cocokkan audit WCC dengan pasangan P1

Pada satu blok P2, nama variabel dibandingkan setelah indeks skenario dilepas. Semua pasangan dicek sampai ke nama dan indeks variabel bersama. Graf penuh mempertahankan indeks skenario pada variabel arus serta menggunakan satu nama untuk setiap keputusan buka gudang.

In [4]:
def coupling(matrix):
    incidence = (matrix != 0).astype(np.int64)
    weights = incidence @ incidence.T
    upper = np.triu(weights, 1)
    return {
        "bobot": int(upper.sum()),
        "pasangan_positif": int(np.count_nonzero(upper)),
        "baris_aktif": int(np.count_nonzero(incidence.any(axis=1))),
        "baris_acuan": int(incidence.shape[0]),
        "variabel": int(incidence.shape[1]),
        "nnz": int(incidence.sum()),
    }


def compare_p1_pairs(a, gy, inbound, outbound, audit, level):
    names = [f"y[{g}]" for g in WAREHOUSES]
    names.extend(f"x[{i},{g},{p}]" for i, g, p in inbound)
    names.extend(f"z[{g},{j},{p}]" for g, j, p in outbound)
    supports = [{names[col] for col in np.flatnonzero(row)} for row in np.c_[gy, a]]
    pairs = list(combinations(range(len(supports)), 2))
    assert len(audit) == len(pairs)
    for (i, j), item in zip(pairs, audit):
        assert (item["baris_1"], item["baris_2"]) == (f"R{i+1}", f"R{j+1}")
        shared = supports[i] & supports[j]
        expected = set(filter(None, item[f"variabel_bersama_{level}"].split("; ")))
        assert shared == expected and len(shared) == int(item[f"bobot_{level}"])


def run():
    p1_pairs = read_p1_pairs()
    probabilities = [item["peluang"] for item in SCENARIOS]
    target = DATA["P2"]["sasaran_permintaan"]
    sigma = DATA["P2"]["simpangan_baku"]
    assert np.isclose(sum(probabilities), 1)
    reference_outbound = [(g, j, p) for g, p in BALANCES for j in CUSTOMERS]
    ar, gyr = route_matrices(HIGH_IN, reference_outbound)
    reference_full, _, _ = expanded_rows(ar, gyr, [{}] * len(SCENARIOS))
    single_ref = coupling(np.c_[gyr, ar])["bobot"]
    full_ref = coupling(reference_full)["bobot"]
    assert (single_ref, full_ref) == (232, 708)
    results = {}
    for level, spec in DATA["level"].items():
        inbound = spec["jalur_masuk"]
        outbound = spec["jalur_keluar"]
        mean_demand = spec["permintaan_dasar"]
        demands = [
            {**mean_demand, target: mean_demand[target] + sigma * scenario["deviasi_baku"]}
            for scenario in SCENARIOS
        ]
        assert min(demand[target] for demand in demands) >= 0
        assert np.isclose(np.dot(probabilities, [d[target] for d in demands]), mean_demand[target])
        a, gy = route_matrices(inbound, outbound)
        compare_p1_pairs(a, gy, inbound, outbound, p1_pairs, level)
        costs = operating_costs(inbound, outbound)
        det_choices = enumerate_warehouses(a, gy, costs, [mean_demand], [1.0])
        stoch_choices = enumerate_warehouses(a, gy, costs, demands, probabilities)
        y_det, det = serialise_best(det_choices)
        _, stoch = serialise_best(stoch_choices)
        det_value, _ = extensive_form(a, gy, costs, [mean_demand], [1.0])
        stoch_value, expanded = extensive_form(a, gy, costs, demands, probabilities)
        assert np.isclose(det_value, det["biaya_objektif"])
        assert np.isclose(stoch_value, stoch["biaya_objektif"])
        base, full = coupling(np.c_[gy, a]), coupling(expanded)
        assert base["bobot"] == {"L":64,"M":97,"H":187}[level]
        assert full["bobot"] == 3 * base["bobot"] + 12
        block_weights = [
            coupling(expanded[39*s:39*(s+1)])["bobot"] for s in range(3)
        ]
        assert block_weights == [base["bobot"]] * 3
        results[level] = {
            "permintaan_K1_A": [scenario[target] for scenario in demands],
            "P1": det,
            "P2": stoch,
            "biaya_y_P1_dalam_P2": stoch_choices[y_det][0] if y_det in stoch_choices else None,
            "audit_satu_blok": {**base, "penyebut":single_ref, "wcc_persen":100*base["bobot"]/single_ref},
            "audit_penuh": {**full, "penyebut":full_ref, "wcc_persen":100*full["bobot"]/full_ref},
        }
    return results

## 5. Baca hasil optimasi dan WCC

In [5]:
results = run()
print("PASS: optimum MILP sama dengan enumerasi dan 741 pasangan per level cocok dengan P1.\n")
print(f"{'Level':<8}{'Gudang P1':<14}{'Biaya P1':>10}{'Gudang P2':>16}{'Biaya P2':>12}"
      f"{'WCC per blok':>16}{'WCC penuh':>14}")
for level, row in results.items():
    p1, p2 = row["P1"], row["P2"]
    warehouse_1 = "/".join(", ".join(ids) for ids in p1["pilihan_optimal"])
    warehouse_2 = "/".join(", ".join(ids) for ids in p2["pilihan_optimal"])
    block = row["audit_satu_blok"]["wcc_persen"]
    full = row["audit_penuh"]["wcc_persen"]
    print(f"{level:<8}{warehouse_1:<14}{p1['biaya_objektif']:>10.4f}{warehouse_2:>16}"
          f"{p2['biaya_objektif']:>12.4f}{block:>15.2f}%{full:>13.2f}%")

PASS: optimum MILP sama dengan enumerasi dan 741 pasangan per level cocok dengan P1.

Level   Gudang P1       Biaya P1       Gudang P2    Biaya P2    WCC per blok     WCC penuh
L       G1, G3, G4      263.9500      G1, G3, G4    263.9500          27.59%        28.81%
M       G2, G3, G4      264.3700      G2, G3, G4    264.4533          41.81%        42.80%
H       G2, G4          242.9500          G2, G3    243.0833          80.60%        80.93%


## 6. Ekspor CSV audit P2 lengkap

CSV memakai kolom yang sama dengan versi P1. ID R36@S1 berarti baris R36 pada skenario Sepi. S2 berarti Sedang dan S3 berarti Ramai. Variabel arus membawa penanda yang sama. Keputusan buka gudang seperti y[1] dipakai bersama semua skenario.

Di Google Colab, hasil CSV muncul pada panel **Files** setelah sel berikut dijalankan.

In [6]:
def full_supports(inbound, outbound):
    a, gy = route_matrices(inbound, outbound)
    matrix, _, _ = expanded_rows(a, gy, [{}] * len(SCENARIOS))
    flow_names = [f"x[{i},{g},{p}]" for i, g, p in inbound]
    flow_names.extend(f"z[{g},{j},{p}]" for g, j, p in outbound)
    names = [f"y[{g}]" for g in WAREHOUSES] + [
        f"{name}@S{s}" for s in range(1, 1 + len(SCENARIOS)) for name in flow_names
    ]
    return [{names[col] for col in np.flatnonzero(row)} for row in matrix]


def export_p2_pairs(path):
    p1_audit = read_p1_pairs()
    header = list(p1_audit[0])
    labels = {}
    for row in p1_audit:
        labels[row["baris_1"]] = row["kendala_1"]
        labels[row["baris_2"]] = row["kendala_2"]
    index = [(row, f"{row}@S{s}") for s in range(1, 1 + len(SCENARIOS))
             for row in (f"R{i}" for i in range(1, 40))]
    reference = full_supports(HIGH_IN, [(g, j, p) for g, p in BALANCES for j in CUSTOMERS])
    actual = {level: full_supports(spec["jalur_masuk"], spec["jalur_keluar"])
              for level, spec in DATA["level"].items()}
    totals = {"acuan": 0, **dict.fromkeys(actual, 0)}
    pair_count = 0
    with Path(path).open("w", newline="", encoding="utf-8") as file:
        writer = DictWriter(file, fieldnames=header, lineterminator="\n")
        writer.writeheader()
        for i, j in combinations(range(len(index)), 2):
            pair_count += 1
            (r, a), (t, b) = index[i], index[j]
            permitted = reference[i] & reference[j]
            totals["acuan"] += len(permitted)
            row = {
                "baris_1": a, "kendala_1": labels[r], "baris_2": b, "kendala_2": labels[t],
                "batas_acuan": len(permitted),
                "variabel_diizinkan_bersama": "; ".join(sorted(permitted)),
            }
            for level, supports in actual.items():
                common = supports[i] & supports[j]
                assert common <= permitted
                totals[level] += len(common)
                row[f"bobot_{level}"] = len(common)
                row[f"variabel_bersama_{level}"] = "; ".join(sorted(common))
            writer.writerow(row)
    assert pair_count == 6786
    assert totals["acuan"] == 708
    assert all(totals[level] == row["audit_penuh"]["bobot"] for level, row in results.items())
    return pair_count, totals


audit_file = "Audit_WCC_Pasangan_P2.csv"
pair_count, totals = export_p2_pairs(audit_file)
print(f"{pair_count:,} pasangan diekspor ke {audit_file}.")
print("Jumlah batas acuan dan bobot tiap level:", totals)

6,786 pasangan diekspor ke Audit_WCC_Pasangan_P2.csv.
Jumlah batas acuan dan bobot tiap level: {'acuan': 708, 'L': 204, 'M': 303, 'H': 573}
